In [ ]:
import os

import pandas as pd
import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

0.環境変数の読み込み

In [ ]:
load_dotenv()
gemini_api_key = os.getenv("API_KEY")

1.chromadbでローカルに作成したdbを読み込む

In [ ]:
client_chromadb = chromadb.PersistentClient(path="./chroma_db")

# 保存済みのコレクションを取得
collection = client_chromadb.get_collection(name="portfolio_rag")


2.embeddingモデルの設定

In [ ]:
# embedding用のモデルを設定
model_embedding = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

3.geminiの設定

In [ ]:
generation_config = {
    "temperature": 0.2,
    "max_output_tokens": 1500, 
}

model_gemini = "gemini-3-flash-preview" 

In [ ]:
client_gemini = genai.Client(api_key = gemini_api_key)

4.RAGパイプライン

In [ ]:
def embedding_query(query):
    return model_embedding.encode(query)


In [ ]:
def query_chromadb(query_embeddings, top_k):

    result = collection.query(
        query_embeddings=[query_embeddings.tolist()],
        n_results=top_k
    )

    context = ""

    for i, (doc, metadata) in enumerate(zip(result["documents"][0], result["metadatas"][0]), start=1):

        context += f"""
            【検索結果{i}】
            source: {metadata["source"]}

            {doc}

            """

    return context

In [ ]:
def make_prompt(query, context):

    prompt = f"""
        あなたはポートフォリオ検索アシスタントです。

        以下の参考情報を基に質問へ回答してください。

        ルール:
        - 回答は自然な日本語で記述すること
        - 箇条書きは必要な場合のみ使用すること
        - 参考情報の見出しや構造をそのまま出力しないこと
        - 「プロジェクト名:」や「source:」などの内部情報は出力しないこと
        - 回答文として読みやすくまとめること
        - 情報が見つからない場合のみ「情報が見つかりませんでした」と回答すること

        参考情報:
        {context}

        質問:
        {query}
    """

    return prompt


In [ ]:
def call_gemini_api(prompt):

    response = client_gemini.models.generate_content(
        model = model_gemini, 
        config = generation_config,
        contents = prompt
    )

    # print(response.text)

    return response.text

In [ ]:
# query = "太陽光発電量予測で使ったモデルは？"
query = "LightGBMを使ったプロジェクトを教えて"
# query = "銀行マーケティングで比較したモデルは？"
# query = "Dockerを利用したプロジェクトは？"

query_embedding = embedding_query(query=query)
context = query_chromadb(query_embeddings=query_embedding, top_k=10)
prompt = make_prompt(query=query, context=context)

response = call_gemini_api(prompt=prompt)

In [ ]:
print(response)

In [ ]:
# rag.pyで関数の型ヒントを書くためのにtypeの確認
# query_embedding = embedding_query(query=query)
print(type(model_embedding))